In [0]:
%run "../includes/configuration"

In [0]:
%run "../includes/common_functions"

In [0]:
dbutils.widgets.text("file_date", "2024-12-16", "File Date")
file_date = dbutils.widgets.get("file_date")

In [0]:
from pyspark.sql.functions import col, current_timestamp, lit
from delta.tables import DeltaTable

In [0]:
movie_cast_df = (
    spark.table("databricks_ws.bronze.movie_casts")
    .filter(col("file_date") == file_date)
    )

In [0]:
movie_cast_selected_df = (
    movie_cast_df
    .select(
        col("movieId").cast("int"),
        col("personId").cast("int"),
        col("characterName").cast("string"),
        col("file_date")
    )
)

In [0]:
movie_cast_with_columns_df = (
    add_ingestion_date(movie_cast_selected_df)
    .withColumnRenamed("movieId", "movie_id")
    .withColumnRenamed("personId", "person_id")
    .withColumnRenamed("characterName", "character_name")
    .withColumn("environment", lit("Production"))
)

In [0]:
table_name = "databricks_ws.silver.movie_casts"
if spark.catalog.tableExists(table_name):

    deltaTable = DeltaTable.forName(spark, table_name)

    deltaTable.alias("tgt") \
        .merge(
            movie_cast_with_columns_df.alias("src"),
            "tgt.movie_id = src.movie_id AND tgt.person_id = src.person_id"
        ) \
        .whenMatchedUpdateAll() \
        .whenNotMatchedInsertAll() \
        .execute()
else:
    movie_cast_with_columns_df.write \
        .format("delta") \
        .saveAsTable(table_name)